In [ ]:
%pip install tqdm scikit-learn pillow pyyaml pandas torchvision torch timm>=0.9.10 huggingface-hub>=0.23.0 open_clip_torch

In [ ]:
import numpy as np
from scipy.special import logsumexp, logit, log_expit
softplusinv = lambda x: np.log(np.expm1(x))  # log(exp(x)-1)
softminusinv = lambda x: x - np.log(-np.expm1(x)) # info: https://jiafulow.github.io/blog/2019/07/11/softplus-and-softminus/

fusion_functions = {
    'mean_logit'   : lambda x, axis: np.mean(x, axis),
    'max_logit'    : lambda x, axis: np.max(x, axis),
    'median_logit' : lambda x, axis: np.median(x, axis),
    'lse_logit'    : lambda x, axis: logsumexp(x, axis),
    'mean_prob'    : lambda x, axis: softminusinv(logsumexp(log_expit(x), axis) - np.log(x.shape[axis])),
    'soft_or_prob' : lambda x, axis: -softminusinv(np.sum(log_expit(-x), axis)),
}

def apply_fusion(x, typ, axis):
    return fusion_functions[typ](x, axis)

In [ ]:
import numbers
import random
from io import BytesIO
import numpy as np
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image


def make_processing(opt):
    # make precessing transform
    # input: an argparse.Namespace
    # output: a torchvision.transforms.Compose
    #

    opt = parse_arguments(opt)
    transforms_list = list()  # list of transforms

    transforms_pre = make_pre(opt)  # make pre-data-augmentation transforms
    if transforms_pre is not None:
        transforms_list.append(transforms_pre)

    transforms_aug = make_aug(opt)  # make data-augmentation transforms
    if transforms_aug is not None:
        idx_aug = len(transforms_list)
        transforms_list.append(transforms_aug)
    else:
        idx_aug = -1

    transforms_post = make_post(opt)  # make post-data-augmentation transforms
    if transforms_post is not None:
        transforms_list.append(transforms_post)

    transforms_list.append(make_normalize(opt.norm_type))  # make normalization

    if (hasattr(opt, "num_views")) and (abs(opt.num_views) > 0):
        print("num_view:", opt.num_views)
        t = transforms.Compose(transforms_list)
        # make multiviews for Self-supervised learning (SSL)
        t = MultiView([t for _ in range(abs(opt.num_views))])
    else:
        t = transforms.Compose(transforms_list)

    return t


def add_processing_arguments(parser):
    # parser is an argparse.ArgumentParser
    #
    # ICASSP2023: --cropSize 96 --loadSize -1 --resizeSize -1 --norm_type resnet --resize_prob 0.2 --jitter_prob 0.8 --colordist_prob 0.2 --cutout_prob 0.2 --noise_prob 0.2 --blur_prob 0.5 --cmp_prob 0.5 --rot90_prob 1.0 --hpf_prob 0.0 --blur_sig 0.0,3.0 --cmp_method cv2,pil --cmp_qual 30,100 --resize_size 256 --resize_ratio 0.75
    # ICME2021  : --cropSize 96 --loadSize -1 --resizeSize -1 --norm_type resnet --resize_prob 0.0 --jitter_prob 0.0 --colordist_prob 0.0 --cutout_prob 0.0 --noise_prob 0.0 --blur_prob 0.5 --cmp_prob 0.5 --rot90_prob 1.0 --hpf_prob 0.0 --blur_sig 0.0,3.0 --cmp_method cv2,pil --cmp_qual 30,100
    #

    parser.add_argument(
        "--resizeSize",
        type=int,
        default=-1,
        help="scale images to this size post augumentation",
    )
    parser.add_argument(
        "--loadSize",
        type=int,
        default=-1,
        help="scale images to this size pre augumentation",
    )
    parser.add_argument(
        "--cropSize",
        type=int,
        default=-1,
        help="crop images to this size post augumentation",
    )
    parser.add_argument("--no_random_crop", action="store_true")

    # data-augmentation probabilities
    parser.add_argument("--resize_prob", type=float, default=0.0)
    parser.add_argument("--jitter_prob", type=float, default=0.0)
    parser.add_argument("--colordist_prob", type=float, default=0.0)
    parser.add_argument("--cutout_prob", type=float, default=0.0)
    parser.add_argument("--noise_prob", type=float, default=0.0)
    parser.add_argument("--blur_prob", type=float, default=0.0)
    parser.add_argument("--cmp_prob", type=float, default=0.0)
    parser.add_argument("--rot90_prob", type=float, default=1.0)
    parser.add_argument("--no_flip", action="store_true")
    parser.add_argument("--hpf_prob", type=float, default=0.0)

    # data-augmentation parameters
    parser.add_argument("--rz_interp", default="bilinear")
    parser.add_argument("--blur_sig", default="0.5")
    parser.add_argument("--cmp_method", default="cv2")
    parser.add_argument("--cmp_qual", default="75")
    parser.add_argument("--resize_size", type=int, default=256)
    parser.add_argument("--resize_ratio", type=float, default=1.0)

    # other
    parser.add_argument("--norm_type", type=str, default="resnet")  # normalization type
    # multi views for Self-supervised learning (SSL)
    parser.add_argument("--num_views", type=int, default=0)

    return parser


def parse_arguments(opt):
    if not isinstance(opt.rz_interp, list):
        opt.rz_interp = list(opt.rz_interp.split(","))
    if not isinstance(opt.blur_sig, list):
        opt.blur_sig = [float(s) for s in opt.blur_sig.split(",")]
    if not isinstance(opt.cmp_method, list):
        opt.cmp_method = list(opt.cmp_method.split(","))
    if not isinstance(opt.cmp_qual, list):
        opt.cmp_qual = [int(s) for s in opt.cmp_qual.split(",")]
        if len(opt.cmp_qual) == 2:
            opt.cmp_qual = list(range(opt.cmp_qual[0], opt.cmp_qual[1] + 1))
        elif len(opt.cmp_qual) > 2:
            raise ValueError("Shouldn't have more than 2 values for --cmp_qual.")
    return opt


rz_dict = {
    "bilinear": Image.BILINEAR,
    "bicubic": Image.BICUBIC,
    "lanczos": Image.LANCZOS,
    "nearest": Image.NEAREST,
}


def make_pre(opt):
    transforms_list = list()
    if opt.loadSize > 0:
        print("\nUsing Pre Resizing\n")
        transforms_list.append(
            transforms.Lambda(
                lambda img: TF.resize(
                    img,
                    opt.loadSize,
                    interpolation=rz_dict[sample_discrete(opt.rz_interp)],
                )
            )
        )
        transforms_list.append(
            CenterCropPad(opt.loadSize, pad_if_needed=True, padding_mode="symmetric")
        )

    if len(transforms_list) == 0:
        return None
    else:
        return transforms.Compose(transforms_list)


def make_post(opt):
    transforms_list = list()
    if opt.resizeSize > 0:
        print("\nUsing Post Resizing\n")
        transforms_list.append(
            transforms.Resize(
                opt.resizeSize, interpolation=transforms.InterpolationMode.BICUBIC
            )
        )
        transforms_list.append(transforms.CenterCrop((opt.resizeSize, opt.resizeSize)))

    if opt.cropSize > 0:
        if not opt.no_random_crop:
            print("\nUsing Post Random Crop\n")
            transforms_list.append(
                transforms.RandomCrop(
                    opt.cropSize, pad_if_needed=True, padding_mode="symmetric"
                )
            )
        else:
            print("\nUsing Post Central Crop\n")
            transforms_list.append(
                CenterCropPad(
                    opt.cropSize, pad_if_needed=True, padding_mode="symmetric"
                )
            )

    if len(transforms_list) == 0:
        return None
    else:
        return transforms.Compose(transforms_list)


def make_aug(opt):
    # AUG
    transforms_list_aug = list()

    if (opt.resize_size > 0) and (opt.resize_prob > 0):  # opt.resized_ratio
        transforms_list_aug.append(
            transforms.RandomApply(
                [
                    transforms.RandomResizedCrop(
                        size=opt.resize_size,
                        scale=(0.08, 1.0),
                        ratio=(opt.resize_ratio, 1.0 / opt.resize_ratio),
                        interpolation=rz_dict[sample_discrete(opt.rz_interp)],
                    )
                ],
                opt.resize_prob,
            )
        )

    if opt.jitter_prob > 0:
        transforms_list_aug.append(
            transforms.RandomApply(
                [transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=opt.jitter_prob
            )
        )

    if opt.colordist_prob > 0:
        transforms_list_aug.append(transforms.RandomGrayscale(p=opt.colordist_prob))

    if opt.cutout_prob > 0:
        transforms_list_aug.append(create_cutout_transforms(opt.cutout_prob))

    if opt.noise_prob > 0:
        transforms_list_aug.append(create_noise_transforms(opt.noise_prob))

    if opt.blur_prob > 0:
        transforms_list_aug.append(
            transforms.Lambda(
                lambda img: data_augment_blur(img, opt.blur_prob, opt.blur_sig)
            )
        )

    if opt.cmp_prob > 0:
        transforms_list_aug.append(
            transforms.Lambda(
                lambda img: data_augment_cmp(
                    img, opt.cmp_prob, opt.cmp_method, opt.cmp_qual
                )
            )
        )

    if opt.rot90_prob > 0:
        transforms_list_aug.append(
            transforms.Lambda(lambda img: data_augment_rot90(img, opt.rot90_prob))
        )

    if opt.hpf_prob > 0:
        transforms_list_aug.append(transforms.ToTensor())
        transforms_list_aug.append(
            transforms.Lambda(
                lambda img: data_augment_hpf(img, opt.hpf_prob, opt.blur_sig)
            )
        )

    if not opt.no_flip:
        transforms_list_aug.append(transforms.RandomHorizontalFlip())

    if len(transforms_list_aug) > 0:
        return transforms.Compose(transforms_list_aug)
    else:
        return None


def make_normalize(norm_type):
    transforms_list = list()

    if norm_type == "resnet":
        print("normalize RESNET")

        transforms_list.append(transforms.ToTensor())
        transforms_list.append(
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        )
    elif norm_type == "clip":
        print("normalize CLIP")
        transforms_list.append(transforms.ToTensor())
        transforms_list.append(
            transforms.Normalize(
                mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711),
            )
        )
    elif norm_type == "xception":
        print("normalize -1,1")

        transforms_list.append(transforms.ToTensor())
        transforms_list.append(
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        )
    elif norm_type == "spec":
        print("normalize SPEC")

        transforms_list.append(normalization_fft)
        transforms_list.append(transforms.ToTensor())

    elif norm_type == "fft2":
        print("normalize Energy")

        transforms_list.append(pic2imgn)
        transforms_list.append(normalization_fft2)
        transforms_list.append(imgn2torch)

    elif norm_type == "residue3":
        print("normalize Residue3")

        transforms_list.append(normalization_residue3)
    elif norm_type == "npr":
        print("normalize NPR")

        transforms_list.append(transforms.ToTensor())
        transforms_list.append(
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        )
        from torch.nn.functional import interpolate
        transforms_list.append(
            lambda x: x[..., :(x.shape[-2]//2*2), :(x.shape[-1]//2*2)]
        )
        transforms_list.append(
            lambda x: (x - interpolate(x[None,...,::2,::2], scale_factor=2.0, mode='nearest', recompute_scale_factor=True)[0])*2.0/3.0
        )
    elif norm_type == "cooc":
        print("normalize COOC")

        transforms_list.append(normalization_cooc)
    else:
        assert False

    return transforms.Compose(transforms_list)


class MultiView:
    def __init__(self, trasfroms_list):
        self.trasfroms_list = trasfroms_list
        print("num_view:", len(self.trasfroms_list))

    def __call__(self, x):
        return torch.stack([fun(x) for fun in self.trasfroms_list], 0)


def sample_discrete(s):
    if len(s) == 1:
        return s[0]
    return random.choice(s)


def sample_continuous(s):
    if len(s) == 1:
        return s[0]
    if len(s) == 2:
        rg = s[1] - s[0]
        return random.random() * rg + s[0]
    raise ValueError("Length of iterable s should be 1 or 2.")


def data_augment_blur(img, p, blur_sig):
    from scipy.ndimage.filters import gaussian_filter
    if random.random() < p:
        img = np.array(img)
        sig = sample_continuous(blur_sig)
        gaussian_filter(img[:, :, 0], output=img[:, :, 0], sigma=sig)
        gaussian_filter(img[:, :, 1], output=img[:, :, 1], sigma=sig)
        gaussian_filter(img[:, :, 2], output=img[:, :, 2], sigma=sig)
        img = Image.fromarray(img)

    return img


def data_augment_hpf(img, p, blur_sig):
    assert isinstance(img, torch.Tensor)
    if random.random() < p:
        sig = 0.4 + sample_continuous(blur_sig)
        kernel_size = int(7 * sig)
        kernel_size = kernel_size + (kernel_size + 1) % 2
        img = img - TF.gaussian_blur(img, kernel_size=kernel_size, sigma=sig)
        img = img + torch.from_numpy(np.asarray([[[0.485]], [[0.456]], [[0.406]]]))
    return img.float()


def data_augment_cmp(img, p, cmp_method, cmp_qual):
    if random.random() < p:
        img = np.array(img)
        method = sample_discrete(cmp_method)
        qual = sample_discrete(cmp_qual)
        img = cmp_from_key(img, qual, method)
        img = Image.fromarray(img)

    return img


def data_augment_rot90(img, p):
    if random.random() < p:
        angle = sample_discrete([0, 90, 180, 270])
        img = img.rotate(angle, expand=True)

    return img


def data_augment_D4(img, p):
    if random.random() < p:
        angle = sample_discrete([0, 90, 180, 270])
        sim = sample_discrete([0, 1])
        img = img.rotate(angle, expand=True)
        if sim == 1:
            img = img.transpose(Image.FLIP_TOP_BOTTOM)
    return img



def create_noise_transforms(p, var_limit=(10.0, 50.0)):
    from albumentations.augmentations.transforms import GaussNoise

    aug = GaussNoise(var_limit=var_limit, always_apply=False, p=p)
    return transforms.Lambda(
        lambda img: Image.fromarray(aug(image=np.array(img))["image"])
    )


def create_cutout_transforms(p):
    try:
        from albumentations.augmentations.dropout.cutout import Cutout
    except:
        from albumentations.augmentations.transforms import Cutout
    aug = Cutout(
        num_holes=1,
        max_h_size=48,
        max_w_size=48,
        fill_value=128,
        always_apply=False,
        p=p,
    )
    return transforms.Lambda(
        lambda img: Image.fromarray(aug(image=np.array(img))["image"])
    )


def cv2_webp(img, compress_val):
    import cv2
    img_cv2 = img[:, :, ::-1]
    encode_param = [int(cv2.IMWRITE_WEBP_QUALITY), compress_val]
    result, encimg = cv2.imencode(".webp", img_cv2, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg[:, :, ::-1]


def pil_webp(img, compress_val):
    out = BytesIO()
    img = Image.fromarray(img)
    img.save(out, format="webp", quality=compress_val)
    img = Image.open(out)
    # load from memory before ByteIO closes
    img = np.array(img)
    out.close()
    return img


def cv2_jpg(img, compress_val):
    import cv2
    img_cv2 = img[:, :, ::-1]
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), compress_val]
    result, encimg = cv2.imencode(".jpg", img_cv2, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg[:, :, ::-1]


def pil_jpg(img, compress_val):
    out = BytesIO()
    img = Image.fromarray(img)
    img.save(out, format="jpeg", quality=compress_val)
    img = Image.open(out)
    # load from memory before ByteIO closes
    img = np.array(img)
    out.close()
    return img


# NOTE: 'cv2' and 'pil' have been left here for legacy reasons
cmp_dict = {
    "cv2": cv2_jpg,
    "cv2_jpg": cv2_jpg,
    "cv2_webp": cv2_webp,
    "pil": pil_jpg,
    "pil_jpg": pil_jpg,
    "pil_webp": pil_webp,
}


def cmp_from_key(img, compress_val, key):
    return cmp_dict[key](img, compress_val)


def pic2imgn(pic):
    from copy import deepcopy

    img = np.float32(deepcopy(np.asarray(pic))) / 256.0
    return img


def imgn2torch(img):
    return torch.from_numpy(img).permute(2, 0, 1).float().contiguous()


def normalization_fft2(img, normalize=512.0):
    img = np.fft.fftshift(np.fft.fft2(img, axes=(0, 1)), axes=(0, 1))
    img = np.square(np.abs(img)) / normalize
    return img


def normalization_fft(pic):
    from copy import deepcopy

    im = np.float32(deepcopy(np.asarray(pic))) / 255.0

    for i in range(im.shape[2]):
        img = im[:, :, i]
        fft_img = np.fft.fft2(img)
        fft_img = np.log(np.abs(fft_img) + 1e-3)
        fft_min = np.percentile(fft_img, 5)
        fft_max = np.percentile(fft_img, 95)
        if (fft_max - fft_min) <= 0:
            print("ma cosa...")
            fft_img = (fft_img - fft_min) / ((fft_max - fft_min) + np.finfo(float).eps)
        else:
            fft_img = (fft_img - fft_min) / (fft_max - fft_min)
        fft_img = (fft_img - 0.5) * 2
        fft_img[fft_img < -1] = -1
        fft_img[fft_img > 1] = 1
        im[:, :, i] = fft_img

    return im


def normalization_residue3(pic, flag_tanh=False):
    from copy import deepcopy

    x = np.float32(deepcopy(np.asarray(pic))) / 32
    wV = (
        -1 * x[1:-3, 2:-2, :]
        + 3 * x[2:-2, 2:-2, :]
        - 3 * x[3:-1, 2:-2, :]
        + 1 * x[4:, 2:-2, :]
    )
    wH = (
        -1 * x[2:-2, 1:-3, :]
        + 3 * x[2:-2, 2:-2, :]
        - 3 * x[2:-2, 3:-1, :]
        + 1 * x[2:-2, 4:, :]
    )
    ress = np.concatenate((wV, wH), -1)
    if flag_tanh:
        ress = np.tanh(ress)

    ress = torch.from_numpy(ress).permute(2, 0, 1).contiguous()

    return ress


def normalization_cooc(pic):
    from copy import deepcopy

    x = deepcopy(np.asarray(pic))
    y = x[1:, 1:, :]
    x = x[:-1, :-1, :]
    bins = np.arange(257)
    H = np.stack(
        [
            np.histogram2d(
                x[:, :, i].flatten(), y[:, :, i].flatten(), bins, density=True
            )[0]
            for i in range(x.shape[2])
        ],
        0,
    )
    H = torch.from_numpy(H)
    return H


class CenterCropPad:
    def __init__(
        self, siz, pad_if_needed=False, padding_fill=0, padding_mode="constant"
    ):
        if isinstance(siz, numbers.Number):
            siz = (int(siz), int(siz))
        self.siz = siz
        self.pad_if_needed = pad_if_needed
        self.padding_fill = padding_fill
        self.padding_mode = padding_mode

    def __call__(self, img):
        crop_height, crop_width = self.siz
        image_width, image_height = img.size[1], img.size[0]
        crop_top = (image_height - crop_height) // 2
        crop_left = (image_width - crop_width) // 2
        if crop_top < 0:
            if self.pad_if_needed:
                img = TF.pad(
                    img,
                    (0, -crop_top, 0, crop_height - image_height + crop_top),
                    fill=self.padding_fill,
                    padding_mode=self.padding_mode,
                )
            else:
                crop_height = image_height
            crop_top = 0
        if crop_left < 0:
            if self.pad_if_needed:
                img = TF.pad(
                    img,
                    (-crop_left, 0, crop_width - image_width + crop_left, 0),
                    fill=self.padding_fill,
                    padding_mode=self.padding_mode,
                )
            else:
                crop_width = image_width
            crop_left = 0
        return img.crop(
            (crop_left, crop_top, crop_left + crop_width, crop_top + crop_height)
        )


In [ ]:
def create_architecture(name_arch, pretrained=False, num_classes=1):
    if name_arch == "res50nodown":
        # from .resnet_mod import resnet50

        if pretrained:
            model = resnet50(pretrained=True, stride0=1, dropout=0.5).change_output(num_classes)
        else:
            model = resnet50(num_classes=num_classes, stride0=1, dropout=0.5)
    elif name_arch == "res50":
        # from .resnet_mod import resnet50

        if pretrained:
            model = resnet50(pretrained=True, stride0=2).change_output(num_classes)
        else:
            model = resnet50(num_classes=num_classes, stride0=2)
    elif name_arch.startswith('opencliplinear_'):
        # from .openclipnet import OpenClipLinear
        model = OpenClipLinear(num_classes=num_classes, pretrain=name_arch[15:], normalize=True)
    elif name_arch.startswith('opencliplinearnext_'):
        # from .openclipnet import OpenClipLinear
        model = OpenClipLinear(num_classes=num_classes, pretrain=name_arch[19:], normalize=True, next_to_last=True)
    else:
        assert False
    return model

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def load_weights(model, model_path):
    from torch import load
    dat = load(model_path, map_location='cpu')
    if 'model' in dat:
        if ('module._conv_stem.weight' in dat['model']) or \
           ('module.fc.fc1.weight' in dat['model']) or \
           ('module.fc.weight' in dat['model']):
            model.load_state_dict(
                {key[7:]: dat['model'][key] for key in dat['model']})
        else:
            model.load_state_dict(dat['model'])
    elif 'state_dict' in dat:
        model.load_state_dict(dat['state_dict'])
    elif 'net' in dat:
        model.load_state_dict(dat['net'])
    elif 'main.0.weight' in dat:
        model.load_state_dict(dat)
    elif '_fc.weight' in dat:
        model.load_state_dict(dat)
    elif 'conv1.weight' in dat:
        model.load_state_dict(dat)
    else:
        print(list(dat.keys()))
        assert False
    return model


In [ ]:
import torch
import torch.nn as nn
import torch.utils.model_zoo as model_zoo

__all__ = ["ResNet", "resnet18", "resnet34", "resnet50", "resnet101", "resnet152"]


model_urls = {
    "resnet18": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    "resnet34": "https://download.pytorch.org/models/resnet34-333f7ec4.pth",
    "resnet50": "https://download.pytorch.org/models/resnet50-19c8e357.pth",
    "resnet101": "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth",
    "resnet152": "https://download.pytorch.org/models/resnet152-b121ed2d.pth",
}

class ChannelLinear(nn.Linear):
    def __init__(
        self, in_features: int, out_features: int, bias: bool = True, pool=None
    ) -> None:
        super(ChannelLinear, self).__init__(in_features, out_features, bias)
        self.compute_axis = 1
        self.pool = pool

    def forward(self, x):
        axis_ref = len(x.shape) - 1
        x = torch.transpose(x, self.compute_axis, axis_ref)
        out_shape = list(x.shape)
        out_shape[-1] = self.out_features
        x = x.reshape(-1, x.shape[-1])
        x = x.matmul(self.weight.t())
        if self.bias is not None:
            x = x + self.bias[None, :]
        x = torch.transpose(x.view(out_shape), axis_ref, self.compute_axis)
        if self.pool is not None:
            x = self.pool(x)
        return x


def conv3x3(in_planes, out_planes, stride=1, padding=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride, padding=padding, bias=False
    )


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, padding=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride, padding=padding)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes, padding=padding)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.padding == 0:
            identity = identity[..., 1:-1, 1:-1]
        if self.downsample is not None:
            identity = self.downsample(identity)
        if self.padding == 0:
            identity = identity[..., 1:-1, 1:-1]

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, padding=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride, padding=padding)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.padding == 0:
            identity = identity[..., 1:-1, 1:-1]
        if self.downsample is not None:
            identity = self.downsample(identity)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(
        self,
        block,
        layers,
        num_classes=1000,
        zero_init_residual=False,
        stride0=2,
        padding=1,
        dropout=0.0,
        gap_size=None,
    ):
        super(ResNet, self).__init__()
        self.inplanes = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size=7, stride=stride0, padding=3 * padding, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        if dropout > 0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = None
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=stride0, padding=padding)
        self.layer1 = self._make_layer(block, 64, layers[0], padding=padding)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, padding=padding)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, padding=padding)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, padding=padding)

        if gap_size is None:
            self.gap_size = None
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        elif gap_size < 0:
            with torch.no_grad():
                y = self.forward_features(
                    torch.zeros((1, 3, -gap_size, -gap_size), dtype=torch.float32)
                ).shape
            print("gap_size:", -gap_size, ">>", y[-1])
            self.gap_size = y[-1]
            self.avgpool = nn.AvgPool2d(kernel_size=self.gap_size, stride=1, padding=0)
        elif gap_size == 1:
            self.gap_size = gap_size
            self.avgpool = None
        else:
            self.gap_size = gap_size
            self.avgpool = nn.AvgPool2d(kernel_size=self.gap_size, stride=1, padding=0)
        self.num_features = 512 * block.expansion
        self.fc = ChannelLinear(self.num_features, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, padding=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes,
                planes,
                stride=stride,
                downsample=downsample,
                padding=padding,
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, padding=padding))

        return nn.Sequential(*layers)

    def change_output(self, num_classes):
        self.fc = ChannelLinear(self.num_features, num_classes)
        torch.nn.init.normal_(self.fc.weight.data, 0.0, 0.02)
        return self

    def change_input(self, num_inputs):
        data = self.conv1.weight.data
        old_num_inputs = int(data.shape[1])
        if num_inputs > old_num_inputs:
            times = num_inputs // old_num_inputs
            if (times * old_num_inputs) < num_inputs:
                times = times + 1
            data = data.repeat(1, times, 1, 1) / times
        elif num_inputs == old_num_inputs:
            return self

        data = data[:, :num_inputs, :, :]
        print(self.conv1.weight.data.shape, "->", data.shape)
        self.conv1.weight.data = data

        return self

    def forward_features(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def forward_head(self, x):
        if self.avgpool is not None:
            x = self.avgpool(x)
        if self.dropout is not None:
            x = self.dropout(x)
        y = self.fc(x)
        if self.gap_size is None:
            y = torch.squeeze(torch.squeeze(y, -1), -1)
        return y

    def forward(self, x):
        x = self.forward_features(x)
        x = self.forward_head(x)
        return x


def resnet18(pretrained=False, **kwargs):
    """Constructs a ResNet-18 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls["resnet18"]))
    return model


def resnet34(pretrained=False, **kwargs):
    """Constructs a ResNet-34 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls["resnet34"]))
    return model


def resnet50(pretrained=False, **kwargs):
    """Constructs a ResNet-50 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls["resnet50"]))
    return model


def resnet101(pretrained=False, **kwargs):
    """Constructs a ResNet-101 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 4, 23, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls["resnet101"]))
    return model


def resnet152(pretrained=False, **kwargs):
    """Constructs a ResNet-152 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 8, 36, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls["resnet152"]))
    return model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
# from .resnet_mod import ChannelLinear

dict_pretrain = {
    'clipL14openai'     : ('ViT-L-14', 'openai'),
    'clipL14laion400m'  : ('ViT-L-14', 'laion400m_e32'),
    'clipL14laion2B'    : ('ViT-L-14', 'laion2b_s32b_b82k'),
    'clipL14datacomp'   : ('ViT-L-14', 'laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K', 'open_clip_pytorch_model.bin'),
    'clipL14commonpool' : ('ViT-L-14', "laion/CLIP-ViT-L-14-CommonPool.XL-s13B-b90K", 'open_clip_pytorch_model.bin'),
    'clipaL14datacomp'  : ('ViT-L-14-CLIPA', 'datacomp1b'),
    'cocaL14laion2B'    : ('coca_ViT-L-14', 'laion2b_s13b_b90k'),
    'clipg14laion2B'    : ('ViT-g-14', 'laion2b_s34b_b88k'),
    'eva2L14merged2b'   : ('EVA02-L-14', 'merged2b_s4b_b131k'),
    'clipB16laion2B'    : ('ViT-B-16', 'laion2b_s34b_b88k'),
}


class OpenClipLinear(nn.Module):
    def __init__(self, num_classes=1, pretrain='clipL14commonpool', normalize=True, next_to_last=False):
        super(OpenClipLinear, self).__init__()
        
        if len(dict_pretrain[pretrain])==2:
            backbone = open_clip.create_model(dict_pretrain[pretrain][0], pretrained=dict_pretrain[pretrain][1])
        else:
            from huggingface_hub import hf_hub_download
            backbone = open_clip.create_model(dict_pretrain[pretrain][0], pretrained=hf_hub_download(*dict_pretrain[pretrain][1:]))
        
        if next_to_last:
            self.num_features = backbone.visual.proj.shape[0]
            backbone.visual.proj = None
        else:
            self.num_features = backbone.visual.output_dim
        
        self.bb = [backbone, ]
        self.normalize = normalize
        
        self.fc = ChannelLinear(self.num_features, num_classes)
        torch.nn.init.normal_(self.fc.weight.data, 0.0, 0.02)

    def to(self, *args, **kwargs):
        self.bb[0].to(*args, **kwargs)
        super(OpenClipLinear, self).to(*args, **kwargs)
        return self

    def forward_features(self, x):
        with torch.no_grad():
            self.bb[0].eval()
            features = self.bb[0].encode_image(x, normalize=self.normalize)
        return features

    def forward_head(self, x):
        return self.fc(x)

    def forward(self, x):
        return self.forward_head(self.forward_features(x))

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # ✅ hiển thị tiến trình

# ==========================================================
# Create commercial_tools.csv from all image folders
# ==========================================================

def create_commercial_tools_csv(
    root_dir="cs406-evaluationrealfake/test",
    output_csv="commercial_tools_val.csv"
):
    image_exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")
    entries = []

    print("🔍 Đang quét thư mục, vui lòng đợi...")
    all_files = []

    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(image_exts):
                all_files.append(os.path.join(dirpath, fname))

    total_files = len(all_files)
    if total_files == 0:
        print("⚠️ Không tìm thấy ảnh nào trong:", root_dir)
        return

    print(f"📸 Tìm thấy {total_files} ảnh. Bắt đầu tạo CSV...\n")

    for full_path in tqdm(all_files, total=total_files, desc="🧩 Đang xử lý", unit="ảnh"):
        rel_path = os.path.relpath(full_path, root_dir)
        parts = rel_path.split(os.sep)

        # parts dạng:
        # real/image.jpg
        # fake/model_name/image.jpg

        if parts[0] == "real":
            typ = "real"
            model = "real"
        elif parts[0] == "fake":
            typ = "fake"
            model = parts[1]  # DALLE, GAN, IF-CC1M, stable_diffusion
        else:
            continue  # nếu có thư mục lạ thì bỏ qua

        entries.append({
            "filename": rel_path,
            "typ": typ,
            "model": model,
            "label": 0 if typ == "real" else 1
        })

    df = pd.DataFrame(entries)
    df.to_csv(output_csv, index=False)
    print(f"\n✅ Hoàn tất! Đã lưu {len(df)} ảnh vào {output_csv}")

# ==========================================================
# Run it
# ==========================================================
create_commercial_tools_csv(
    root_dir="/kaggle/input/cs406-evaluationrealfake/test", 
    output_csv="/kaggle/working/commercial_tools.csv"
)

In [ ]:
import torch
import os
import pandas
import numpy as np
import tqdm
import glob
import sys
import yaml
from PIL import Image

from torchvision.transforms  import CenterCrop, Resize, Compose, InterpolationMode
# from utils.processing import make_normalize
# from utils.fusion import apply_fusion
# from networks import create_architecture, load_weights


def get_config(model_name, weights_dir='./weights'):
    with open(os.path.join(weights_dir, model_name, 'config.yaml')) as fid:
        data = yaml.load(fid, Loader=yaml.FullLoader)
    model_path = os.path.join(weights_dir, model_name, data['weights_file'])
    return data['model_name'], model_path, data['arch'], data['norm_type'], data['patch_size']


def runnig_tests(input_csv, weights_dir, models_list, device,root_data, batch_size = 1):
    table = pandas.read_csv(input_csv)[['filename',]]
    rootdataset = root_data
    
    models_dict = dict()
    transform_dict = dict()
    print("Models:")
    for model_name in models_list:
        print(model_name, flush=True)
        _, model_path, arch, norm_type, patch_size = get_config(model_name, weights_dir=weights_dir)

        model = load_weights(create_architecture(arch), model_path)
        model = model.to(device).eval()

        transform = list()
        if patch_size is None:
            print('input resize: default 256', flush=True)
            transform.append(Resize(256, interpolation=InterpolationMode.BICUBIC))
            transform.append(CenterCrop(256))
            transform_key = 'res256_%s' % norm_type
        elif patch_size=='Clip224':
            print('input resize:', 'Clip224', flush=True)
            transform.append(Resize(224, interpolation=InterpolationMode.BICUBIC))
            transform.append(CenterCrop((224, 224)))
            transform_key = 'Clip224_%s' % norm_type
        elif isinstance(patch_size, tuple) or isinstance(patch_size, list):
            print('input resize:', patch_size, flush=True)
            transform.append(Resize(*patch_size))
            transform.append(CenterCrop(patch_size[0]))
            transform_key = 'res%d_%s' % (patch_size[0], norm_type)
        elif patch_size > 0:
            print('input crop:', patch_size, flush=True)
            transform.append(CenterCrop(patch_size))
            transform_key = 'crop%d_%s' % (patch_size, norm_type)
        
        transform.append(make_normalize(norm_type))
        transform = Compose(transform)
        transform_dict[transform_key] = transform
        models_dict[model_name] = (transform_key, model)
        print(flush=True)

    ### test
    with torch.no_grad():
        
        do_models = list(models_dict.keys())
        do_transforms = set([models_dict[_][0] for _ in do_models])
        print(do_models)
        print(do_transforms)
        print(flush=True)
        
        print("Running the Tests")
        batch_img = {k: list() for k in transform_dict}
        batch_id = list()
        last_index = table.index[-1]
        for index in tqdm.tqdm(table.index, total=len(table)):
            filename = os.path.join(rootdataset, table.loc[index, 'filename'])
            # load ảnh 1 lần
            try:
                img = Image.open(filename)
                img.load()
                img = img.convert("RGB")
            except Exception as e:
                print(f"⚠️ Skipping corrupted image: {filename} ({e})")
                continue  # bỏ qua index này hoàn toàn
            
            # nếu ảnh load được → apply tất cả transform
            ok = True
            for k in transform_dict:
                try:
                    batch_img[k].append(transform_dict[k](img))
                except Exception as e:
                    print(f"⚠️ Transform failed on: {filename} ({e})")
                    ok = False
                    break
            
            # nếu có lỗi transform → reset và bỏ index
            if not ok:
                for k in transform_dict:
                    if len(batch_img[k])>0:
                        batch_img[k].pop()
                continue
                
            batch_id.append(index)

            if (len(batch_id) >= batch_size) or (index==last_index):
                for k in do_transforms:
                    batch_img[k] = torch.stack(batch_img[k], 0)

                for model_name in do_models:
                    out_tens = models_dict[model_name][1](batch_img[models_dict[model_name][0]].to(device)).cpu().numpy()

                    if out_tens.shape[1] == 1:
                        out_tens = out_tens[:, 0]
                    elif out_tens.shape[1] == 2:
                        out_tens = out_tens[:, 1] - out_tens[:, 0]
                    else:
                        assert False
                    
                    if len(out_tens.shape) > 1:
                        logit1 = np.mean(out_tens, (1, 2))
                    else:
                        logit1 = out_tens

                    for ii, logit in zip(batch_id, logit1):
                        table.loc[ii, model_name] = logit

                batch_img = {k: list() for k in transform_dict}
                batch_id = list()
                
            torch.cuda.empty_cache()

            assert len(batch_id)==0
        
    return table

In [ ]:
# ==========================================================
# Chạy trực tiếp trên Kaggle mà không cần parser CLI
# ==========================================================

# 🧩 Giả lập các tham số từ dòng lệnh
in_csv      = "/kaggle/working/commercial_tools.csv"
out_csv     = "/kaggle/working/out.csv"
weights_dir = "/kaggle/input/clip-sythetic-image-weights/weights"
models      = "clipdet_latent10k_plus,Corvi2023"
fusion      = "soft_or_prob"
device      = "cuda:0" if torch.cuda.is_available() else "cpu"
root_data   = "/kaggle/input/cs406-evaluationrealfake/test"

# 🧩 Xử lý giá trị models
if models is None:
    models = os.listdir(weights_dir)
else:
    models = models.split(',')

# 🧩 Gọi trực tiếp hàm như bạn đang gọi trong main
table = runnig_tests(in_csv, weights_dir, models, device,root_data)
# input_csv, weights_dir, models_list, device, batch_size = 1

if fusion is not None:
    table['fusion'] = apply_fusion(table[models].values, fusion, axis=-1)
# Nhãn dự đoán: logit > 0 → fake
table['pred_label'] = (table['fusion'] > 0).astype(int)

# Confidence: sigmoid(logit)
table['confidence'] = 1 / (1 + np.exp(-table['fusion']))

# 🧩 Lưu kết quả ra file
os.makedirs(os.path.dirname(os.path.abspath(out_csv)), exist_ok=True)
table.to_csv(out_csv, index=False)
print(f"✅ Saved results to: {out_csv}")

# 🧩 Hiển thị một vài dòng đầu tiên
table.head()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn import metrics as skmetrics  # ✅ đổi alias để tránh nhầm

# ==========================================================
# Metric dictionary
# ==========================================================
dict_metrics = {
    'auc' : lambda label, score: skmetrics.roc_auc_score(label,  score),
    'acc' : lambda label, score: skmetrics.balanced_accuracy_score(label, score > 0),
}

# ==========================================================
# Function
# ==========================================================
def compute_metrics(input_csv, output_csv, metrics_fun):
    table = pd.read_csv(output_csv)
    list_algs = [c for c in table.columns if c != 'filename']

    table = pd.read_csv(input_csv).merge(table, on=['filename'])
    assert 'typ' in table

    list_typs = sorted([t for t in set(table['typ']) if t != 'real'])
    table['label'] = table['typ'] != 'real'

    tab_metrics = pd.DataFrame(index=list_algs, columns=list_typs)
    tab_metrics.loc[:, :] = np.nan

    for typ in list_typs:
        tab_typ = table[table['typ'].isin(['real', typ])]
        for alg in list_algs:
            score = tab_typ[alg].values
            label = tab_typ['label'].values
            if not np.all(np.isfinite(score)):
                continue
            tab_metrics.loc[alg, typ] = metrics_fun(label, score)

    tab_metrics['AVG'] = tab_metrics.mean(1)
    return tab_metrics

In [ ]:
import os
import pandas as pd

# ==========================================================
# 🧩 Thiết lập tham số giống như argparse nhưng chạy trực tiếp
# ==========================================================
in_csv   = "/kaggle/working/commercial_tools.csv"   # CSV input gốc
out_csv  = "/kaggle/working/out.csv"           # CSV kết quả mô hình
metrics  = "auc"                                    # hoặc "acc"
save_tab = "/kaggle/working/metrics_auc_results.csv"    # nơi lưu metrics (hoặc None)

# ==========================================================
# 🧩 Gọi trực tiếp compute_metrics như trong main
# ==========================================================
tab_metrics = compute_metrics(in_csv, out_csv, dict_metrics[metrics])
tab_metrics.index.name = metrics

# Hiển thị bảng kết quả
print(tab_metrics.to_string(float_format=lambda x: '%5.3f' % x))

# ==========================================================
# 🧩 Lưu ra file nếu cần
# ==========================================================
if save_tab is not None:
    os.makedirs(os.path.dirname(os.path.abspath(save_tab)), exist_ok=True)
    tab_metrics.to_csv(save_tab, index=True)
    print(f"✅ Metrics saved to {save_tab}")